# CHRUTH - Pipeline unique

Ce notebook est le point d'entrée principal pour générer les livrables CHRUTH.

Il pilote le moteur `CHRUTH_PIPELINE_UNIQUE.py`, qui régénère :
- le cockpit appels d'offres `output/AO_CHRUTH.xlsm` ;
- la base prospects `output/Base_Prospects_CHRUTH.xlsm` ;
- la carte `output/Carte_Prospects_CHRUTH.html` ;
- le CRM, les KPI, les exports Notion/Power BI ;
- le modèle financier ;
- les documents de mission et prompts ;
- un dossier portable si demandé.

Par défaut, la collecte réseau est désactivée : le notebook retraite les données locales.

## 1. Installer les dépendances

À exécuter une fois sur un nouveau poste. Si tout est déjà installé, cette cellule ne change rien d'important.

In [ ]:
%pip install -q -r requirements.txt

## 2. Régler les options

- `COLLECTE_AO` : recollecte BOAMP/DCE.
- `COLLECTE_PROSPECTS` : recollecte API Entreprises, potentiellement longue.
- `GENERER_MESSAGES` : active la génération de brouillons par segment si un LLM est disponible.
- `CREER_PACK` : copie le dossier en version portable prête à envoyer.
- `SCOPE_PROSPECTS` : `france`, `region`, `departements` ou `test`.

In [ ]:
COLLECTE_AO = False
COLLECTE_PROSPECTS = False
GENERER_MESSAGES = False
CREER_PACK = True

SCOPE_PROSPECTS = "france"
REGIONS = ""
DEPARTEMENTS = "69"

SKIP_AO = False
SKIP_PROSPECTS = False
SKIP_FINANCE = False

## 3. Lancer toute la pipeline

Ferme les fichiers Excel ouverts dans `output/` avant de lancer si tu veux que les classeurs soient réécrits.

In [ ]:
import subprocess
import sys
from pathlib import Path

cmd = [sys.executable, "CHRUTH_PIPELINE_UNIQUE.py"]
if COLLECTE_AO:
    cmd.append("--collect-ao")
if COLLECTE_PROSPECTS:
    cmd.extend(["--collect-prospects", "--scope", SCOPE_PROSPECTS])
    if SCOPE_PROSPECTS == "region" and REGIONS.strip():
        cmd.extend(["--regions", REGIONS])
    if SCOPE_PROSPECTS == "departements":
        cmd.extend(["--departements", DEPARTEMENTS])
elif REGIONS.strip():
    cmd.extend(["--regions", REGIONS])
if SKIP_AO:
    cmd.append("--skip-ao")
if SKIP_PROSPECTS:
    cmd.append("--skip-prospects")
if SKIP_FINANCE:
    cmd.append("--skip-finance")
if GENERER_MESSAGES:
    cmd.append("--generer-messages")
if CREER_PACK:
    cmd.append("--pack")

print("Commande:", " ".join(cmd))
result = subprocess.run(cmd, cwd=Path.cwd(), text=True)
if result.returncode != 0:
    raise SystemExit(result.returncode)

## 4. Contrôler les sorties

Cette cellule liste les livrables principaux attendus.

In [ ]:
from pathlib import Path

attendus = [
    "output/AO_CHRUTH.xlsm",
    "output/Base_Prospects_CHRUTH.xlsm",
    "output/Carte_Prospects_CHRUTH.html",
    "output/CRM_CHRUTH_CHAUDE.xlsx",
    "output/Prospects_CHAUDS_messages.xlsx",
    "output/Modele_Financier_CHRUTH.xlsx",
    "output/MANIFEST_CHRUTH.json",
    "output/LIRE_MOI_LIVRABLES.md",
    "docs/MISSION_CHRUTH.md",
    "prompts/PROMPTS_CHRUTH.md",
]
for item in attendus:
    path = Path(item)
    print(("OK   " if path.exists() else "MISS ") + item)

## 5. Lecture rapide

- `output/LIRE_MOI_LIVRABLES.md` explique les fichiers générés.
- `docs/MISSION_CHRUTH.md` relie les sorties aux missions de la fiche de poste.
- `prompts/PROMPTS_CHRUTH.md` regroupe les prompts utiles.
- Le dossier portable est créé dans `Downloads/CHRUTH_LIVRAISON_UNIFIEE_YYYYMMDD_HHMM` si `CREER_PACK=True`.